In [ ]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

import xarrams as xrr

from metpy.units import units
import metpy.constants as mpconstants
import metpy.calc as mpc

In [ ]:
generated_sounding = xrr.soundings.wk84_sounding(
    U_s=5 * units("m/s"), q_v0=11 * units("g/kg"), p_sfc=100000 * units("Pa")
)
generated_sounding = xrr.soundings.calculate_sounding_derived_vars(generated_sounding)
# generated_sounding["dewpoint"] = mpc.dewpoint_from_relative_humidity(
#     temperature=generated_sounding["TS"].values * units("degC"),
#     relative_humidity=generated_sounding["RTS"].values * units("percent"),
# )
display(generated_sounding.head(30))
display(generated_sounding.tail(30))
xrr.soundings.plot_sounding_skewt(generated_sounding, barbs=False)

In [ ]:
cm1_sounding_ds = xr.open_dataset("../data/cm1_wk_sounding_qv-11.nc")
cm1_sounding_ds

In [ ]:
cm1_sounding_df = cm1_sounding_ds.to_pandas().reset_index()
cm1_sounding_df["PS"] = cm1_sounding_ds["prs"].values / 100.0

cm1_sounding_df["TS"] = (
    mpc.temperature_from_potential_temperature(
        pressure=cm1_sounding_df["PS"].values * units("hPa"),
        potential_temperature=cm1_sounding_df["th"].values * units("K"),
    )
    .to("degC")
    .magnitude
)

vp = mpc.vapor_pressure(
    cm1_sounding_df["PS"].values * units("hPa"),
    cm1_sounding_df["qv"].values * units("kg/kg"),
)
cm1_sounding_df["dewpoint"] = mpc.dewpoint(vp).to("degC").magnitude
cm1_sounding_df["z"] = cm1_sounding_ds["zh"].values
cm1_sounding_df["US"] = 0
cm1_sounding_df["VS"] = 0

xrr.soundings.plot_sounding_skewt(cm1_sounding_df, barbs=False)

display(cm1_sounding_df)

In [ ]:
# Create a comparison
# Interpolate our sounding to the levels in CM1
basic_state_vars = ["z", "TS", "PS", "dewpoint"]
# generated_sounding_interped =
generated_sounding_interped = pd.DataFrame({"z": cm1_sounding_df["z"].values})
interp_cols = [c for c in basic_state_vars if c != "z"]
for c in interp_cols:
    generated_sounding_interped[c] = np.interp(
        generated_sounding_interped["z"].values,
        generated_sounding["z"].values,
        generated_sounding[c].values,
    )

to_merge = [
    generated_sounding_interped[basic_state_vars],
    cm1_sounding_df[basic_state_vars],
]

for df in to_merge:
    df["theta"] = (
        mpc.potential_temperature(
            pressure=df["PS"].values * units("hPa"),
            temperature=df["TS"].values * units("degC"),
        )
        .to("K")
        .magnitude
    )
    df["RTS"] = (
        mpc.relative_humidity_from_dewpoint(
            temperature=df["TS"].values * units("degC"),
            dewpoint=df["dewpoint"].values * units("degC"),
        )
        .to("percent")
        .magnitude
    )
    df["qv"] = (
        mpc.mixing_ratio_from_relative_humidity(
            pressure=df["PS"].values * units("hPa"),
            temperature=df["TS"].values * units("degC"),
            relative_humidity=df["RTS"].values * units("percent"),
        )
        .to("g/kg")
        .magnitude
    )
    df["parcel_T"] = mpc.parcel_profile(
        pressure=df["PS"].values * units("hPa"),
        temperature=df["TS"].values[0] * units("degC"),
        dewpoint=df["dewpoint"].values[0] * units("degC"),
    )
    df["parcel_theta"] = mpc.potential_temperature(
        pressure=df["PS"].values * units("hPa"),
        temperature=df["parcel_T"].values * units("K"),
    )
    integrand = (df["parcel_theta"] - df["theta"]) / df["theta"]
    # Restrict to positive-buoyancy region (the "positive area on a skew-T")
    positive = np.where(integrand > 0, integrand, 0.0)

    integrand_da = xr.DataArray(
        # ((parcel_path_theta.to('K').magnitude - sounding['THETA']) / sounding['THETA']),
        integrand,
        dims=["z"],
        coords={"z": df["z"].values},
    )
    g = mpconstants.g.to("m/s^2").magnitude
    B = (integrand_da.where(integrand_da > 0, 0).integrate("z") * g).item()
    print(B)

merged_df = pd.merge(*to_merge, on="z", suffixes=("_g", "_cm1"))

theta_0 = 300
theta_tr = 343
T_tr = 213
z_tr = 12000

merged_df["theta_calc"] = np.where(
    df["z"] <= z_tr,
    (theta_0 + (theta_tr - theta_0) * (df["z"] / z_tr) ** (5 / 4)),
    theta_tr * np.exp((9.8 / (1004 * T_tr)) * (df["z"] - z_tr)),
)

merged_df = merged_df[["z"] + sorted([x for x in merged_df.columns if x != "z"])]
display(merged_df)

fig, axs = plt.subplots(ncols=2, figsize=(8, 3), layout="constrained", sharey=True)

axs[0].plot(
    merged_df["TS_g"] - merged_df["TS_cm1"],
    merged_df["z"].values,
)
axs[0].set_title("T deviation from CM1")

axs[1].plot(
    merged_df["dewpoint_g"] - merged_df["dewpoint_cm1"],
    merged_df["z"].values,
)
axs[1].set_title("Dewpoint deviation from CM1")